# Stage 7 — Scored Differential Diagnosis

LLM produces a **ranked, scored** differential for the **current** admission from:

1. **Symptom tree** (`symptom_tree.json`)
2. **Retained SNOMED context** (`snomed_retained.json`)
3. **Prior admission ICDs** — **all** billed ICD-10 codes from `admission_history.json` (PMH / comorbidity only)
4. Optional: structured clinical context + IE (current stay)

**Prompt format:** ROLE / CONTEXT / TASK / CONSTRAINTS  
**Temperature:** `0.4`  
**Does not** use current-stay ground-truth ICD or discharge diagnosis lists.

**Input:** Stage 4–6 exports under `patient_records/`  
**Output:** `data/stage_07_differential_diagnosis/` + per-admission `differential_diagnosis.json` / `.txt`


In [ ]:
import json
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"
sys.path.insert(0, str(NB_DIR))

from pipeline import (
    DIFF_DX_CHECKPOINT_JSON,
    DIFF_DX_RESULTS_JSON,
    DIFF_DX_TEMPERATURE,
    EXPORT_DIR,
    LLMNotAvailableError,
    LLM_REQUEST_DELAY_SECONDS,
    STAGE_07_DIR,
    build_prior_icd_context,
    check_llm,
    differential_diagnosis_agent,
    export_diff_dx_to_admission,
    get_llm_config,
    list_admission_export_dirs,
    load_diff_dx_checkpoint,
    print_pipeline_banner,
    save_diff_dx_checkpoint,
    save_diff_dx_results,
    warn_if_slow_model,
)

print_pipeline_banner()
LLM_CONFIG = get_llm_config()
ok, model_info = check_llm(LLM_CONFIG)
if not ok:
    raise LLMNotAvailableError(model_info)
warn_if_slow_model(model_info, LLM_CONFIG.provider)
print(f"LLM ready — {LLM_CONFIG.method_prefix()}: {model_info}")
print(f"DiffDx temperature: {DIFF_DX_TEMPERATURE}")
print(f"Prompt format: ROLE / CONTEXT / TASK / CONSTRAINTS (+ all prior ICDs)")

STAGE_07_DIR.mkdir(parents=True, exist_ok=True)
print(f"Export dir : {EXPORT_DIR}")
print(f"Stage 7 out: {STAGE_07_DIR}")


In [ ]:
admissions = list_admission_export_dirs(EXPORT_DIR)
ready = [a for a in admissions if a["has_symptom_tree"]]
missing_tree = [a for a in admissions if not a["has_symptom_tree"]]
missing_ret = [a for a in ready if not a["has_retained"]]
missing_hist = []
for a in ready:
    hp = EXPORT_DIR / f"patient_{a['patient_id']}" / "admission_history.json"
    if not hp.exists():
        missing_hist.append(a["patient_id"])
print(f"Admissions found     : {len(admissions)}")
print(f"With symptom tree    : {len(ready)}")
print(f"Missing retained SNOMED (run with empty): {len(missing_ret)}")
print(f"Missing admission_history.json: {len(set(missing_hist))}")
if missing_tree:
    print("Skip (no tree):", [(a["patient_id"], a["hadm_id"]) for a in missing_tree[:5]])


In [ ]:
done = load_diff_dx_checkpoint()
records = list(done.values())
todo = [
    a for a in ready
    if f"{a['patient_id']}|{a['hadm_id']}" not in done
]
print(f"Resuming: {len(done)} done, {len(todo)} remaining")
print("Note: delete diff_dx_checkpoint.json to re-run all with the new prior-ICD prompt.\n")

for i, adm in enumerate(todo, start=1):
    pid, hid = adm["patient_id"], adm["hadm_id"]
    adm_dir = Path(adm["admission_dir"])
    print(f"[{i}/{len(todo)}] DiffDx patient={pid} hadm={hid}...")

    tree = json.loads(Path(adm["symptom_tree_path"]).read_text(encoding="utf-8"))
    retained = {}
    if adm["has_retained"]:
        retained = json.loads(Path(adm["retained_path"]).read_text(encoding="utf-8"))

    # Prior admissions — ALL ICD-10 codes (Option B)
    hist_path = EXPORT_DIR / f"patient_{pid}" / "admission_history.json"
    history = []
    if hist_path.exists():
        history = json.loads(hist_path.read_text(encoding="utf-8"))
        if not isinstance(history, list):
            history = []
    prior_preview = build_prior_icd_context(history)
    n_icds = sum(len(p.get("icd10_diagnoses") or []) for p in prior_preview)
    print(f"  prior admissions={len(prior_preview)} | prior ICD codes={n_icds}")

    ctx_path = adm_dir / "clinical_context.txt"
    clinical_context = ctx_path.read_text(encoding="utf-8") if ctx_path.exists() else None

    ie = None
    ie_path = adm_dir / "information_extraction.json"
    if ie_path.exists():
        ie = json.loads(ie_path.read_text(encoding="utf-8"))

    try:
        result = differential_diagnosis_agent(
            symptom_tree=tree,
            retained_snomed=retained,
            patient_id=pid,
            hadm_id=hid,
            clinical_context_text=clinical_context,
            ie_summary=ie,
            admission_history=history,
            config=LLM_CONFIG,
            temperature=DIFF_DX_TEMPERATURE,
        )
    except (ValueError, TimeoutError, LLMNotAvailableError) as exc:
        print(f"  ERROR: {exc}")
        result = {
            "patient_id": pid,
            "hadm_id": hid,
            "type": "differential_diagnosis",
            "error": str(exc),
            "differential": [],
            "n_candidates": 0,
            "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
        }

    export_diff_dx_to_admission(result, adm_dir)
    records.append(result)
    save_diff_dx_checkpoint(records)

    top = (result.get("differential") or [{}])[0]
    print(
        f"  most_likely={result.get('most_likely')!r} | "
        f"n={result.get('n_candidates')} | "
        f"top_score={top.get('score')} | "
        f"temp={result.get('_temperature')}"
    )

    if i < len(todo) and LLM_REQUEST_DELAY_SECONDS > 0:
        time.sleep(LLM_REQUEST_DELAY_SECONDS)

out = save_diff_dx_results(records)
print(f"\nSaved aggregate → {out}")
print(f"Per-admission: {EXPORT_DIR}/patient_*/admissions/hadm_*/differential_diagnosis.*")


In [ ]:
# Preview first successful run
for row in records:
    if row.get("differential"):
        print(f"Patient {row.get('patient_id')} HADM {row.get('hadm_id')}")
        print(f"Temp={row.get('_temperature')} | inputs={row.get('inputs')}")
        print(f"Most likely: {row.get('most_likely')}")
        print(f"Summary: {row.get('summary', '')[:400]}")
        print("\nDifferential:")
        for d in row["differential"]:
            print(
                f"  #{d['rank']}  {d['score']:>5}/100  {d['diagnosis']}  "
                f"[{d.get('confidence')}|{d.get('category')}]"
            )
        break
else:
    print("No differentials in results yet.")
